# Figure 2B Slingshot trajectories

Use CVG neuroblasts as the root without specifying endpoints. Start from the saved embryonic CSS integration and retain all 7,355 cells, their original UMAP coordinates and annotations, and 393 reviewed markers.

The native analysis runs Slingshot `getLineages` and `getCurves` on the first ten standardized CSS principal components. Membership weights are balanced by study, stage and state with exponent 0.5. The curve settings are df=5, reweight=False and reassign=False; the native `embedCurves(df=4)` projection is also saved. The final display uses the selected MST paths and early CVG branch. Endpoint-constrained smoothing centers the displayed shared trunk without changing native pseudotime or lineage weights.

Among the 100 and 200 nearest cells around the SGN fork, both modal and median ages are E14.5; E13.5-E14.5 account for 64% and 63.5%, respectively. These neighborhoods do not establish an exact branching date. E9.5 remains the root, and the early fork belongs to the E11.5 proliferative CVG state. Ic lies along the Ib-related path; this model does not have four independent SGN terminal leaves.

Run the sections in order. Set `RSCRIPT` to your Rscript executable when refitting; R requires slingshot and jsonlite (original versions: R 4.5.1 and slingshot 2.18.0). With existing results, `RECOMPUTE=False` loads them directly. After changing analysis or trunk parameters, set it to True and rerun from the input section. Appearance-only changes require the settings, saved-data and plotting sections, without R or refitting.

Save `Slingshot-2.rds` and `Slingshot-2.h5ad` under `ROOT`. They retain the native model, projections, display geometry, metadata, principal components, original UMAP, pseudotime, weights, distances, curves, MST and parameters. RNA counts remain in the integration input.

Export four cell/stage label/legend views as 600-dpi PNGs and editable PDFs under `ROOT`. PDF text uses Arial; points and paths remain vectors. Label PDFs have three sibling clipping groups and legend PDFs have four, with no raster images or Form XObjects.

The original coordinates, 7% padding, point appearance and seed-42 draw order match the Monocle3 display. Label and legend canvases are 8.6 x 8.3 and 11.2 x 7.7 inches. Native pseudotime is unchanged; negative ticks use complete ASCII strings.


In [ ]:
#1. =========== Paths and shared settings
from pathlib import Path
import os,io,json,hashlib,tempfile,subprocess,shutil
# Replace each ROOT placeholder with the required file or directory path before running.
ROOT=Path(".")
if ROOT.name=='Code_def':ROOT=ROOT.parent
# Uses CSS_3000_Embryonic_Inte_adata.h5ad generated by fig1_Embryonic_integration.ipynb.
INPUT=ROOT
# Set the file or directory path.
RESULT_DIR=ROOT
# Uses Slingshot-2.h5ad generated by the root-only Slingshot fitting step.
RESULT_FILE=ROOT
# Uses Slingshot-2.rds generated by the root-only Slingshot fitting step.
MODEL_FILE=ROOT
# Set the file or directory path.
FIGURES=ROOT
# Set the executable or tool directory path.
R_SCRIPT=ROOT
SEED,N_THREADS=0,1
for key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS'):os.environ[key]=str(N_THREADS)
# Set the temporary parent directory; this run gets its own removable folder.
TEMP_PARENT=Path(tempfile.mkdtemp(prefix="slingshot-", dir=ROOT))
TEMP_PARENT.mkdir(parents=True,exist_ok=True)
_runtime=tempfile.TemporaryDirectory(prefix='run-',dir=TEMP_PARENT)
WORK=Path(_runtime.name)
# Set the file or directory path.
os.environ['MPLCONFIGDIR']=str(ROOT)
import numpy as np,pandas as pd,anndata as ad
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits
from scipy.spatial import cKDTree
from scipy.interpolate import BSpline
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt,matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from IPython.display import display,Image
ad.settings.allow_write_nullable_strings=True
CELL_COLORS={'CVG neuroblast': '#A05837', 'Differentiating SGN': '#006FA6', 'Emerging Type Ic': '#3B9700', 'Immature Type II SGN': '#B79762', 'Immature Type Ia SGN': '#00846F', 'Immature Type Ib SGN': '#00489C', 'Immature Type Ic SGN': '#CC0744', 'Late CVG': '#6B7900', 'O-2: delaminating otic/CVG neuroblast': '#004D43', 'O-3: developing CVG neuron (not SGN-specific)': '#CB7E98', 'Type Ia + Type II precursor candidate': '#BA0900', 'Type Ib + Type Ic precursor candidate': '#A77500', 'Undifferentiated / proliferative CVG': '#B903AA', 'Undifferentiated SGN': '#008941'}
STAGE_COLORS={'E9.5': '#D157A0', 'E10.5': '#CC0744', 'E11.5': '#3B9700', 'E13.5': '#3B5DFF', 'E14.5': '#D16100', 'E15.5': '#00846F', 'E16.5': '#BC23FF', 'E17.5': '#6A3A4C', 'E18.5': '#9B9700'}


In [ ]:
#2. =========== Integration input and selected parameters
RECOMPUTE=False             # Set True to refit this model without endpoint priors.
N_CSS_PCS,N_TRAJECTORY_PCS=30,10
ROOT_CLUSTER='CVG neuroblast'
BALANCE_POWER=.5            # Balance membership weights by study, stage and state.
NATIVE_DF,UMAP_DF=5,4
APPROX_POINTS,MAX_ITERATIONS=200,40
CONVERGENCE=.001
TRUNK_INTERNAL_KNOTS,TRUNK_CURVATURE_PENALTY,TRUNK_POINTS=7,30.0,350
OBS_FIELDS=['cell_type_plot','stage_standardized','leiden','study_id','library_id','GSM',
    'original_cell_id','source_cell_id','original_annotation','integration_batch',
    'source_review_flag','source_rna_mixture_review_flag','source_state_evaluation_allowed','author_SGN_tree_member']
params=dict(root=ROOT_CLUSTER,endpoints=None,balance=BALANCE_POWER,distance='simple',median=False,
    df=NATIVE_DF,view_df=UMAP_DF,approx_points=APPROX_POINTS,maxit=MAX_ITERATIONS,
    thresh=CONVERGENCE,shrink=1,stretch=0,extend='n',reweight=False,reassign=False,
    allow_breaks=True,shrink_method='cosine',seed=SEED,css_pcs=N_CSS_PCS,trajectory_pcs=N_TRAJECTORY_PCS)
display_params=dict(internal_knots=TRUNK_INTERNAL_KNOTS,curvature_penalty=TRUNK_CURVATURE_PENALTY,points=TRUNK_POINTS)
with INPUT.open('rb') as handle:input_sha=hashlib.file_digest(handle,'sha256').hexdigest()
assert input_sha=='8625e64dfb80313661f2258c1363e63ebe3e6f5446c4e2da3c920e46137e1e2c', 'Use the selected embryonic integration file.'
needs_fit=RECOMPUTE or not (RESULT_FILE.is_file() and MODEL_FILE.is_file())
a=ad.read_h5ad(INPUT,backed='r')
obs=a.obs[OBS_FIELDS].copy()
xy=np.asarray(a.obsm['X_umap'],dtype=float).copy()
if needs_fit:
    with threadpool_limits(limits=N_THREADS):
        pcs=PCA(n_components=N_CSS_PCS,svd_solver='full',random_state=SEED).fit_transform(a.obsm['X_css'])[:,:N_TRAJECTORY_PCS]
    white10=pcs/pcs.std(axis=0,ddof=1)
a.file.close()
assert len(obs)==7355


In [ ]:
#3. =========== Fit Slingshot, centre the shared trunk and save two results
# Set the paths used by the embedded R step.
R_SOURCE=r'''ROOT <- "."  # Set the file paths before running this R step.
suppressPackageStartupMessages(library(slingshot))
suppressPackageStartupMessages(library(jsonlite))
work <- normalizePath(commandArgs(trailingOnly=TRUE)[1], winslash="/", mustWork=TRUE)
# Uses parameters.json generated by the trajectory input preparation step.
cfg <- fromJSON(ROOT, simplifyVector=FALSE)
# Uses obs.csv generated by the Slingshot input preparation step.
obs <- read.csv(ROOT, row.names=1, check.names=FALSE)
# Uses white10.csv generated by the Slingshot input preparation step.
rd <- as.matrix(read.csv(ROOT, row.names=1, check.names=FALSE))
# Uses umap.csv generated by the trajectory input preparation step.
umap <- as.matrix(read.csv(ROOT, row.names=1, check.names=FALSE))
stopifnot(identical(rownames(rd), rownames(obs)), identical(rownames(umap), rownames(obs)))
set.seed(cfg$seed)
cl <- as.character(obs$cell_type_plot)
strata <- paste(cl, obs$study_id, obs$stage_standardized, sep="__")
stratum_n <- table(strata)
w <- pmin(1, 20/as.numeric(stratum_n[strata]))^cfg$balance
groups <- unique(cl)
fitcl <- vapply(groups, function(k) as.numeric(cl==k)*w, rep(0, length(cl)))
colnames(fitcl) <- groups
fit <- getLineages(rd, fitcl, start.clus=cfg$root, end.clus=NULL,
                   dist.method=cfg$distance, omega=FALSE, use.median=cfg$median)
fit <- getCurves(fit, approx_points=cfg$approx_points, maxit=cfg$maxit,
                 thresh=cfg$thresh, shrink=cfg$shrink, stretch=cfg$stretch,
                 extend=cfg$extend, smoother="smooth.spline", df=cfg$df,
                 reweight=cfg$reweight, reassign=cfg$reassign,
                 shrink.method=cfg$shrink_method, allow.breaks=cfg$allow_breaks)
view <- embedCurves(fit, umap, approx_points=cfg$approx_points, stretch=cfg$stretch,
                    shrink=cfg$shrink, smoother="smooth.spline", df=cfg$view_df,
                    shrink.method=cfg$shrink_method)
# Uses pseudotime.csv generated by the Slingshot R fitting step.
write.csv(slingPseudotime(fit), ROOT, quote=FALSE)
# Uses weights.csv generated by the Slingshot R fitting step.
write.csv(slingCurveWeights(fit), ROOT, quote=FALSE)
# Uses distances.csv generated by the Slingshot R fitting step.
write.csv(sapply(slingCurves(fit), function(x) x$dist_ind), ROOT, quote=FALSE)
for (j in seq_along(slingCurves(fit))) {
    native <- slingCurves(fit)[[j]]
    projection <- slingCurves(view)[[j]]
    # Uses native/UMAP curve CSV files generated by the Slingshot R fitting step.
    write.csv(native$s[native$ord,,drop=FALSE], ROOT, row.names=FALSE)
    # Uses native/UMAP curve CSV files generated by the Slingshot R fitting step.
    write.csv(projection$s[projection$ord,,drop=FALSE], ROOT, row.names=FALSE)
}
versions <- list(R=as.character(getRversion()), slingshot=as.character(packageVersion("slingshot")))
# Uses native.rds generated by the root-only Slingshot R fitting step.
saveRDS(list(native=fit, umap=view, config=cfg, versions=versions), ROOT)
write_json(list(root=cfg$root, lineages=unname(slingLineages(fit)),
                edges=unname(igraph::as_edgelist(slingMST(fit))), versions=versions),
           # Uses model_info.json generated by the Slingshot R fitting step.
           ROOT, auto_unbox=TRUE, digits=15)
cat("Fitted root-only Slingshot: ", nrow(rd), " cells, ", length(slingLineages(fit)), " lineages.\n", sep="")
'''
if needs_fit:
    # Uses white10.csv generated by the Slingshot input preparation step.
    pd.DataFrame(white10,index=obs.index).to_csv(ROOT)
    # Uses umap.csv generated by the trajectory input preparation step.
    pd.DataFrame(xy,index=obs.index).to_csv(ROOT)
    # Uses obs.csv generated by the Slingshot input preparation step.
    obs[['cell_type_plot','study_id','stage_standardized']].to_csv(ROOT)
    # Uses parameters.json generated by the trajectory input preparation step.
    (ROOT).write_text(json.dumps(params),encoding='utf-8')
    # Set the file or directory path.
    (ROOT).write_text(R_SOURCE,encoding='utf-8')
    env=os.environ.copy();env.update(LC_ALL='C',TMP=str(WORK),TEMP=str(WORK))
    # Set the file or directory path.
    completed=subprocess.run([R_SCRIPT,str(ROOT),str(WORK)],capture_output=True,text=True,env=env)
    if completed.returncode:raise RuntimeError(completed.stderr)
    # Uses model_info.json generated by the Slingshot R fitting step.
    info=json.loads((ROOT).read_text())
    cl=obs.cell_type_plot.astype(str)
    strata=cl+'__'+obs.study_id.astype(str)+'__'+obs.stage_standardized.astype(str)
    case_w=np.minimum(1,20/strata.map(strata.value_counts()).to_numpy())**BALANCE_POWER
    centers={key:np.average(xy[cl.eq(key)],axis=0,weights=case_w[cl.eq(key)]) for key in np.unique(cl)}
    paths=[p for p in info['lineages'] if 'Undifferentiated SGN' in p]
    shared=[]
    for items in zip(*paths):
        if len(set(items))!=1:break
        shared.append(items[0])
    assert shared[-1]=='Undifferentiated SGN'
    trunk_states=shared[1:]
    trunk_pairs={frozenset((u,v)) for u,v in zip(trunk_states[:-1],trunk_states[1:])}
    start=centers[trunk_states[0]];branch=centers[trunk_states[-1]]
    lo,hi=float(branch[0]),float(start[0])
    mask=cl.isin(trunk_states).to_numpy() & (xy[:,0]>=lo) & (xy[:,0]<=hi)
    cloud=xy[mask];w=case_w[mask]
    knots=np.r_[[lo]*4,np.linspace(lo,hi,TRUNK_INTERNAL_KNOTS+2)[1:-1],[hi]*4]
    n_basis=len(knots)-4
    basis=BSpline(knots,np.eye(n_basis),3)
    design=basis(cloud[:,0]);grid=np.linspace(lo,hi,400)
    second=basis.derivative(2)(grid)
    penalty=second.T@second*(hi-lo)/(len(grid)-1)
    normal=design.T@(w[:,None]*design)+TRUNK_CURVATURE_PENALTY*penalty
    constraints=basis(np.array([lo,hi]))
    system=np.block([[normal,constraints.T],[constraints,np.zeros((2,2))]])
    coeff=np.linalg.solve(system,np.r_[design.T@(w*cloud[:,1]),branch[1],start[1]])[:n_basis]
    trunk_x=np.linspace(hi,lo,TRUNK_POINTS)
    trunk=np.column_stack((trunk_x,basis(trunk_x)@coeff))
    trunk[0]=start;trunk[-1]=branch
    fixed_indices=[i for i,(u,v) in enumerate(info['edges']) if frozenset((u,v)) not in trunk_pairs]
    plotted=[np.vstack((centers[u],centers[v])) for i,(u,v) in enumerate(info['edges']) if i in fixed_indices]+[trunk]
    ages=obs.stage_standardized.str[1:].astype(float).to_numpy()
    age_checks=[]
    for k in (100,200):
        near=cKDTree(xy).query(branch,k=k)[1]
        counts=obs.iloc[near].stage_standardized.value_counts().to_dict()
        age_checks.append(dict(neighbors=k,counts={key:int(value) for key,value in counts.items()},
            median_stage=float(np.median(ages[near])),E13_5_E14_5_fraction=float(np.isin(ages[near],[13.5,14.5]).mean())))
    assert all(c['median_stage']==14.5 and c['counts']['E14.5']==max(c['counts'].values()) for c in age_checks)
    node_names=list(centers)
    view_data=dict(method='Original MST branches plus endpoint-constrained centred shared trunk; display only',
        parameters_json=json.dumps(display_params,sort_keys=True),
        node_names=np.asarray(node_names),node_xy=np.stack([centers[k] for k in node_names]),
        mst_edges=np.asarray(info['edges']),frozen_edge_indices=np.asarray(fixed_indices),
        start=start,SGN_branch=branch,trunk=trunk,
        plot_paths={f'path{i:02}':p for i,p in enumerate(plotted)},stage_neighborhoods_json=json.dumps(age_checks),
        endpoint_priors=False,original_pseudotime_preserved=True,
        interpretation='E14.5-enriched SGN branch neighborhood, not an exact split date; E9.5 root and E11.5 early CVG fork.')
    r_display={k:(v.tolist() if isinstance(v,np.ndarray) else v) for k,v in view_data.items() if k!='plot_paths'}
    r_display['plot_paths']=[p.tolist() for p in plotted]
    # Uses display.json generated by the root-only Slingshot display step.
    (ROOT).write_text(json.dumps(dict(display=r_display,input_sha256=input_sha,
        input_file=str(INPUT.relative_to(ROOT)),result_file=RESULT_FILE.name,
        palettes=dict(cell=CELL_COLORS,stage=STAGE_COLORS))),encoding='utf-8')
    # Set the paths used by the embedded R step.
    save_r=r'''ROOT <- "."  # Set the file paths before running this R step.
suppressPackageStartupMessages(library(jsonlite))
work<-commandArgs(trailingOnly=TRUE)[1]
# Uses native.rds generated by the root-only Slingshot R fitting step.
obj<-readRDS(ROOT)
# Uses display.json generated by the root-only Slingshot display step.
extra<-fromJSON(ROOT,simplifyVector=FALSE)
obj<-c(obj,extra)
# Uses Slingshot-2.rds generated by the root-only Slingshot fitting step.
saveRDS(obj,ROOT,compress='xz')
'''
    # Set the file or directory path.
    (ROOT).write_text(save_r,encoding='utf-8')
    # Set the file or directory path.
    completed=subprocess.run([R_SCRIPT,str(ROOT),str(WORK)],capture_output=True,text=True,env=env)
    if completed.returncode:raise RuntimeError(completed.stderr)
    result=ad.AnnData(obs=obs.copy())
    result.obsm['X_umap']=xy
    result.obsm['X_slingshot_input']=white10
    for name in ('pseudotime','weights','distances'):
        # Set the file or directory path.
        frame=pd.read_csv(ROOT,index_col=0)
        assert frame.index.astype(str).equals(obs.index.astype(str))
        result.obsm['slingshot_'+name]=frame.to_numpy()
    lineage_names=np.asarray([f'Lineage{i+1}' for i in range(len(info['lineages']))])
    result.uns['slingshot']=dict(parameters_json=json.dumps(params,sort_keys=True),
        input_file=str(INPUT.relative_to(ROOT)),input_sha256=input_sha,
        lineage_names=lineage_names,lineages={key:np.asarray(p) for key,p in zip(lineage_names,info['lineages'])},
        # Uses native/UMAP curve CSV files generated by the Slingshot R fitting step.
        curves_native=np.stack([pd.read_csv(ROOT).to_numpy() for i in range(len(lineage_names))]),
        # Uses native/UMAP curve CSV files generated by the Slingshot R fitting step.
        curves_umap=np.stack([pd.read_csv(ROOT).to_numpy() for i in range(len(lineage_names))]),
        palettes_json=json.dumps(dict(cell=CELL_COLORS,stage=STAGE_COLORS)),
        software_json=json.dumps(info['versions']),model_file=MODEL_FILE.name,
        # Uses Slingshot-2.rds generated by the root-only Slingshot fitting step.
        model_sha256=hashlib.sha256((ROOT).read_bytes()).hexdigest(),
        endpoint_priors=False,full_biological_acceptance=False)
    result.uns['display']=view_data
    # Uses Slingshot-2.h5ad generated by the root-only Slingshot fitting step.
    result.write_h5ad(ROOT,compression='gzip',convert_strings_to_categoricals=False)
    RESULT_DIR.mkdir(parents=True,exist_ok=True)
    # Uses Slingshot-2.rds generated by the root-only Slingshot fitting step.
    shutil.copyfile(ROOT,MODEL_FILE)
    # Uses Slingshot-2.h5ad generated by the root-only Slingshot fitting step.
    shutil.copyfile(ROOT,RESULT_FILE)
else:
    result=ad.read_h5ad(RESULT_FILE)
saved=result.uns['slingshot']
assert saved['input_sha256']==input_sha and saved['parameters_json']==json.dumps(params,sort_keys=True), 'Input or parameters changed; set RECOMPUTE=True.'
assert result.uns['display']['parameters_json']==json.dumps(display_params,sort_keys=True), 'Display parameters changed; set RECOMPUTE=True.'
assert hashlib.sha256(MODEL_FILE.read_bytes()).hexdigest()==saved['model_sha256']
pd.testing.assert_frame_equal(result.obs,obs)
np.testing.assert_array_equal(result.obsm['X_umap'],xy)
print(f'Saved result ready: {result.n_obs:,} cells, four root-only lineages, two reusable files.')


In [ ]:
#4. =========== Load saved results for plotting without R
result=ad.read_h5ad(RESULT_FILE)
obs=result.obs
xy=result.obsm['X_umap']
saved=result.uns['slingshot']
view_data=result.uns['display']
model=dict(lineages=[list(saved['lineages'][key]) for key in saved['lineage_names']],
    plot_paths=[view_data['plot_paths'][key] for key in sorted(view_data['plot_paths'])])


In [ ]:
#5. =========== Four final figures: PNG and editable PDF
POINT_SIZE,POINT_ALPHA,CURVE_WIDTH=1.35,.82,1.05
LABEL_SIZE,LEGEND_SIZE=8.6,8.0
LEADER_WIDTH,LEADER_COLOR,LABEL_FONT=.90,'#666666','Arial'
DPI,INLINE_DPI=600,130
# Match the selected Monocle3 canvas, square viewport and original UMAP.
# Float32 padding matches 03_moncle3-1.ipynb exactly; no cells are cropped.
view_xy=np.asarray(xy,dtype=np.float32)
lo,hi=view_xy.min(axis=0),view_xy.max(axis=0)
pad=np.maximum((hi-lo)*.07,.3)
X_LIMITS,Y_LIMITS=[(float(lo[j]-pad[j]),float(hi[j]+pad[j])) for j in range(2)]
X_TICKS=(-5,0,5,10,15)
Y_TICKS=(-5,-2.5,0,2.5,5,7.5,10,12.5)
FIGURE_SIZES={'label':(8.6,8.3),'legend':(11.2,7.7)}
AXES_RECTS={'label':(.12,.11,.80,.80),'legend':(.08,.12,.54,.80)}
POINT_ORDER=np.random.default_rng(42).permutation(len(obs))
from matplotlib import font_manager
from matplotlib.ticker import FuncFormatter
# Set the file or directory path.
font_manager.fontManager.addfont(str(ROOT))
plt.rcdefaults()
plt.rcParams.update({'font.family':'Arial','figure.facecolor':'white',
                     'axes.grid':False,'axes.unicode_minus':False})
TITLE='Slingshot 2 | root only\nOriginal branches + centred shared trunk'
CELL_LAYOUT=[
    {'text': 'Late CVG', 'key': 'Late CVG', 'position': [17.5, -2.8], 'anchor': [15.935911178588867, -1.2458840608596802], 'leader': True, 'role': 'intermediate'},
    {'text': 'Developing CVG', 'key': 'O-3: developing CVG neuron (not SGN-specific)', 'position': [13.6, -5.0], 'anchor': [13.850686073303223, -1.7633548974990845], 'leader': True, 'role': 'intermediate'},
    {'text': 'Differentiating SGN', 'key': 'Differentiating SGN', 'position': [8.8, -6.1], 'anchor': [9.416763305664062, -2.8386173248291016], 'leader': True, 'role': 'intermediate'},
    {'text': 'Undifferentiated SGN', 'key': 'Undifferentiated SGN', 'position': [1.0, -2.5], 'anchor': [4.631065845489502, -1.2171224355697632], 'leader': True, 'role': 'intermediate'},
    {'text': 'Ib/Ic precursor\ncandidate cells', 'key': 'Type Ib + Type Ic precursor candidate', 'position': [7.6, 3.7], 'anchor': [4.101393222808838, 1.4937266111373901], 'leader': True, 'role': 'intermediate'},
    {'text': 'Emerging Ic', 'key': 'Emerging Type Ic', 'position': [-5.5, 1.9], 'anchor': [-1.9976530075073242, 3.3447935581207275], 'leader': True, 'role': 'intermediate'},
    {'text': 'Immature Ic', 'key': 'Immature Type Ic SGN', 'position': [-5.6, 4.0], 'anchor': [-2.160126209259033, 4.0376715660095215], 'leader': True, 'role': 'intermediate'},
    {'text': 'CVG neuroblast', 'key': 'CVG neuroblast', 'position': [16.9, 2.05], 'anchor': [16.93683433532715, 0.8245120644569397], 'leader': False, 'role': 'start'},
    {'text': 'Delaminating\notic/CVG', 'key': 'O-2: delaminating otic/CVG neuroblast', 'position': [14.4, 0.85], 'anchor': [16.22874641418457, 0.8835893273353577], 'leader': False, 'role': 'terminal'},
    {'text': 'Immature Ia', 'key': 'Immature Type Ia SGN', 'position': [-2.4, 12.9], 'anchor': [-1.4836064577102661, 10.677833557128906], 'leader': False, 'role': 'terminal'},
    {'text': 'Immature II', 'key': 'Immature Type II SGN', 'position': [5.4, 10.2], 'anchor': [4.072268486022949, 9.853074073791504], 'leader': False, 'role': 'terminal'},
    {'text': 'Immature Ib', 'key': 'Immature Type Ib SGN', 'position': [-5.6, 7.3], 'anchor': [-3.949798107147217, 6.628283500671387], 'leader': False, 'role': 'terminal'},
    {'text': 'Proliferative CVG', 'key': 'Undifferentiated / proliferative CVG', 'position': [18.1, -0.35], 'anchor': [16.7603702545166, -0.034635450690984726], 'leader': False, 'role': 'branch'},
    {'text': 'Ia/II precursor\ncandidate cells', 'key': 'Type Ia + Type II precursor candidate', 'position': [2.95, 7.7], 'anchor': [0.8441885709762573, 8.875995635986328], 'leader': False, 'role': 'branch'},
    {'text': 'Ia/II candidate\nbranch', 'key': 'branch:Ia/II', 'position': [3.4, 5.4], 'anchor': [2.252101274620288, 4.226526776776366], 'leader': False, 'role': 'branch', 'fontsize': 10.0, 'fontweight': 'bold', 'annotation_type': 'inferred_branch_direction'},
    {'text': 'Ib/Ic candidate\nbranch', 'key': 'branch:Ib/Ic', 'position': [-1.0, 1.6], 'anchor': [0.9337147956821382, 2.8390684532768096], 'leader': False, 'role': 'branch', 'fontsize': 10.0, 'fontweight': 'bold', 'annotation_type': 'inferred_branch_direction'},
]
STAGE_LAYOUT=[
    {'text': 'E10.5', 'key': 'E10.5', 'position': [14.1, 1.0], 'anchor': [16.408246994018555, 0.5825856924057007], 'leader': True, 'role': 'intermediate'},
    {'text': 'E11.5', 'key': 'E11.5', 'position': [18.5, -2.2], 'anchor': [16.753856658935547, -0.5795614719390869], 'leader': True, 'role': 'intermediate'},
    {'text': 'E13.5', 'key': 'E13.5', 'position': [12.2, -6.0], 'anchor': [11.06688404083252, -3.4347176551818848], 'leader': True, 'role': 'intermediate'},
    {'text': 'E14.5', 'key': 'E14.5', 'position': [6.7, -5.2], 'anchor': [6.961612224578857, -3.156460762023926], 'leader': True, 'role': 'intermediate'},
    {'text': 'E15.5', 'key': 'E15.5', 'position': [7.2, 2.25], 'anchor': [4.101393222808838, 1.4937266111373901], 'leader': True, 'role': 'intermediate'},
    {'text': 'E16.5', 'key': 'E16.5', 'position': [5.6, 7.5], 'anchor': [1.4477938413619995, 7.970571994781494], 'leader': True, 'role': 'intermediate'},
    {'text': 'E16.5', 'key': 'E16.5', 'position': [-5.8, 3.0], 'anchor': [-1.6518501043319702, 4.451498985290527], 'leader': True, 'role': 'intermediate'},
    {'text': 'E17.5', 'key': 'E17.5', 'position': [3.6, 13.5], 'anchor': [2.527223587036133, 10.29873275756836], 'leader': True, 'role': 'intermediate'},
    {'text': 'E9.5', 'key': 'E9.5', 'position': [16.9, 2.05], 'anchor': [16.93683433532715, 0.8245120644569397], 'leader': False, 'role': 'start'},
    {'text': 'E18.5', 'key': 'E18.5', 'position': [-2.5, 13.4], 'anchor': [-2.441587209701538, 11.622183799743652], 'leader': False, 'role': 'terminal'},
    {'text': 'E18.5', 'key': 'E18.5', 'position': [5.2, 10.3], 'anchor': [4.3119425773620605, 9.730276107788086], 'leader': False, 'role': 'terminal'},
    {'text': 'E18.5', 'key': 'E18.5', 'position': [-5.9, 7.4], 'anchor': [-4.224396228790283, 7.545580863952637], 'leader': False, 'role': 'terminal'},
    {'text': 'E14.5-rich\nSGN split', 'key': 'branch:SGN', 'position': [6.4, 0.6], 'anchor': [3.855947844691269, 0.9920374673554037], 'leader': False, 'role': 'branch', 'fontsize': 8.6, 'fontweight': 'normal'},
]

from scipy.interpolate import CubicSpline
from matplotlib import colors as mcolors,markers as mmarkers
from matplotlib.path import Path as MplPath
from matplotlib.text import Text,Annotation
from matplotlib.collections import PathCollection
from matplotlib.patches import Patch
from reportlab.pdfgen import canvas as rl_canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

def editable_pdf(fig, output):
    """Keep paths and semantic text objects editable; no images or Form XObjects."""
    original_fonts=[(a,a.get_fontproperties().copy()) for a in fig.findobj(lambda x:isinstance(x,Text))]
    for artist,_ in original_fonts:artist.set_fontfamily('Arial')
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    unit = 72 / fig.dpi
    width, height = fig.get_size_inches() * 72
    # Set the file or directory path.
    for name, filename in [('Arial', str(ROOT)), ('ArialBold', str(ROOT))]:
        if name not in pdfmetrics.getRegisteredFontNames():
            # Set the file or directory path.
            pdfmetrics.registerFont(TTFont(name, str(ROOT)))
    data = io.BytesIO()
    pdf = rl_canvas.Canvas(data, pagesize=(width, height), pageCompression=1, invariant=1, initialFontName='Arial')
    pdf.setTitle(Path(output).stem)
    pdf.setCreator('Slingshot vector export')

    def color(value, stroke=False, alpha=None):
        rgba = mcolors.to_rgba(value)
        (pdf.setStrokeColorRGB if stroke else pdf.setFillColorRGB)(*rgba[:3])
        (pdf.setStrokeAlpha if stroke else pdf.setFillAlpha)(rgba[3] if alpha is None else alpha)

    def path(vertices, codes=None):
        p = pdf.beginPath()
        previous = None
        for points, code in MplPath(vertices, codes).iter_segments(curves=True, simplify=False):
            v = points.reshape(-1, 2) * unit
            if code in (MplPath.CURVE3, MplPath.CURVE4) and previous is not None:
                direction, controls = v[-1] - previous, v[:-1] - previous
                crosses = controls[:, 0] * direction[1] - controls[:, 1] * direction[0]
                along = controls @ direction
                if np.allclose(crosses, 0, atol=1e-8) and np.all(along >= 0) and np.all(along <= direction @ direction):
                    v, code = v[-1:], MplPath.LINETO
            if code == MplPath.MOVETO:
                p.moveTo(*v[0])
            elif code == MplPath.LINETO:
                p.lineTo(*v[0])
            elif code == MplPath.CURVE3:
                q, endpoint = v
                c1 = previous + 2 / 3 * (q - previous)
                c2 = endpoint + 2 / 3 * (q - endpoint)
                p.curveTo(*c1, *c2, *endpoint)
            elif code == MplPath.CURVE4:
                p.curveTo(*v.ravel())
            elif code == MplPath.CLOSEPOLY:
                p.close()
            if len(v):
                previous = v[-1]
        return p

    def draw(artist):
        if not artist.get_visible():
            return
        pdf.saveState()
        if isinstance(artist, Text):
            if not artist.get_text():
                pdf.restoreState()
                return
            position = artist.get_transform().transform(artist.get_position())
            layout = artist._get_layout(renderer)[1]
            angle = artist.get_rotation()
            font = 'ArialBold' if artist.get_fontweight() in ('bold', 'heavy', 700, 800, 900) else 'Arial'
            color(artist.get_color())
            text = pdf.beginText()
            text.setFont(font, artist.get_fontsize())
            theta = np.deg2rad(angle)
            c, s = np.cos(theta), np.sin(theta)
            for line, _, x, y in layout:
                origin = (position + [x, y]) * unit
                text.setTextTransform(c, s, -s, c, *origin)
                text.textOut(line)
            pdf.drawText(text)
        elif isinstance(artist, PathCollection):
            centers = artist.get_offset_transform().transform(artist.get_offsets()) * unit
            sizes = artist.get_sizes()
            faces, edges = artist.get_facecolors(), artist.get_edgecolors()
            linewidths = artist.get_linewidths()
            for i, (x, y) in enumerate(centers):
                if not np.isfinite(x + y):
                    continue
                fill = len(faces) and faces[i % len(faces), 3] > 0
                stroke = len(edges) and edges[i % len(edges), 3] > 0 and linewidths[i % len(linewidths)] > 0
                if fill:
                    color(faces[i % len(faces)])
                if stroke:
                    color(edges[i % len(edges)], stroke=True)
                    pdf.setLineWidth(linewidths[i % len(linewidths)])
                marker = artist.get_paths()[i % len(artist.get_paths())]
                vertices = marker.vertices*np.sqrt(sizes[i % len(sizes)])/unit + np.array([x,y])/unit
                pdf.drawPath(path(vertices,marker.codes),stroke=int(stroke),fill=int(fill))
        elif isinstance(artist, Line2D):
            color(artist.get_color(), stroke=True, alpha=artist.get_alpha())
            pdf.setLineWidth(artist.get_linewidth())
            pdf.setLineJoin(1)
            points = artist.get_transform().transform(artist.get_xydata())
            if artist.get_linestyle() not in ('None', 'none', '', ' '):
                if artist.get_linestyle() in ('--', ':', '-.'):
                    pdf.setDash({'--': [3, 2], ':': [1, 1.5], '-.': [3, 2, 1, 2]}[artist.get_linestyle()])
                if len(points)>2:
                    # Interpolate the saved smooth path with editable cubic Bezier segments.
                    derivative=CubicSpline(np.arange(len(points)),points)(np.arange(len(points)),1)
                    vertices=[points[0]];codes=[MplPath.MOVETO]
                    for j in range(len(points)-1):
                        vertices.extend((points[j]+derivative[j]/3,points[j+1]-derivative[j+1]/3,points[j+1]))
                        codes.extend((MplPath.CURVE4,)*3)
                    pdf.drawPath(path(np.asarray(vertices),codes),stroke=1,fill=0)
                else:
                    pdf.drawPath(path(points),stroke=1,fill=0)
            if artist.get_marker() not in ('None', 'none', '', ' ', None):
                marked = points
                every = artist.get_markevery()
                if isinstance(every, (int, np.integer)):
                    marked = points[::every]
                elif isinstance(every, tuple):
                    marked = points[every[0]::every[1]]
                elif every is not None:
                    marked = points[every]
                marker = mmarkers.MarkerStyle(artist.get_marker())
                mp = marker.get_path().transformed(marker.get_transform())
                scale = artist.get_markersize() / unit
                color(artist.get_markeredgecolor(), stroke=True)
                pdf.setLineWidth(artist.get_markeredgewidth())
                face = artist.get_markerfacecolor()
                fill = mcolors.to_rgba(face)[3] > 0
                if fill:
                    color(face)
                for point in marked:
                    if artist.get_marker() == 'o':
                        pdf.circle(*(point * unit), artist.get_markersize() / 2, stroke=1, fill=int(fill))
                    else:
                        pdf.drawPath(path(mp.vertices * scale + point, mp.codes), stroke=1, fill=int(fill))
        elif isinstance(artist, Patch):
            color(artist.get_edgecolor(), stroke=True)
            color(artist.get_facecolor())
            pdf.setLineWidth(artist.get_linewidth())
            p = artist.get_path().transformed(artist.get_transform())
            pdf.drawPath(path(p.vertices, p.codes), stroke=int(artist.get_linewidth() > 0 and artist.get_edgecolor()[3] > 0),
                         fill=int(artist.get_fill() and artist.get_facecolor()[3] > 0))
        pdf.restoreState()

    groups = {name: [] for name in ('Graphics', 'Axes/Guides', 'Text', 'Legend')}
    for ax in fig.axes:
        groups['Axes/Guides'].extend(ax.spines.values())
        for axis in (ax.xaxis, ax.yaxis):
            for tick in axis._update_ticks():
                groups['Axes/Guides'].extend((tick.tick1line,tick.tick2line,tick.gridline))
                groups['Text'].extend((tick.label1,tick.label2))
            groups['Text'].extend((axis.label,axis.get_offset_text()))
        groups['Text'].extend([ax.title,ax._left_title,ax._right_title,*ax.texts])
        groups['Graphics'].extend([*ax.collections,*ax.lines,*ax.patches])
        # Annotation leader lines are separate open vector paths, not part of the text.
        groups['Axes/Guides'].extend(t.arrow_patch for t in ax.texts
            if isinstance(t,Annotation) and t.arrow_patch is not None)
        legend = ax.get_legend()
        if legend:
            groups['Legend'].extend(legend.get_texts())
            groups['Legend'].extend(legend.findobj(lambda a:isinstance(a,(PathCollection,Line2D,Patch))))
    groups['Text'].extend(fig.texts)
    for group_name,artists in groups.items():
        artists=[a for a in {id(a):a for a in artists}.values() if a.get_visible()]
        if not artists:continue
        pdf.saveState()
        pdf._code.append('/%s BMC' % group_name.replace('/','-'))
        rectangle = pdf.beginPath()
        rectangle.rect(0,0,width,height)
        pdf.clipPath(rectangle,stroke=0,fill=0,fillMode=1)
        for artist in sorted({id(a):a for a in artists}.values(),key=lambda a:a.get_zorder()):
            draw(artist)
        pdf._code.append('EMC')
        pdf.restoreState()
    pdf.showPage()
    pdf.save()
    Path(output).write_bytes(data.getvalue())
    for artist,properties in original_fonts:artist.set_fontproperties(properties)
    fig.canvas.draw()


def place_labels(ax,specs):
    texts=[]
    for s in specs:
        style=dict(ha='center',va='center',fontsize=s.get('fontsize',LABEL_SIZE),
            fontweight=s.get('fontweight','normal'),fontfamily=LABEL_FONT,zorder=6,color='black')
        if s['leader']:
            text=ax.annotate(s['text'],xy=s['anchor'],xytext=s['position'],**style,
                arrowprops=dict(arrowstyle='-',lw=LEADER_WIDTH,color=LEADER_COLOR,alpha=1,shrinkA=5,shrinkB=2))
        else:text=ax.text(*s['position'],s['text'],**style)
        text.set_path_effects([pe.withStroke(linewidth=1.7,foreground='white')]);texts.append(text)
    ax.figure.canvas.draw();renderer=ax.figure.canvas.get_renderer()
    boxes=[matplotlib.text.Text.get_window_extent(t,renderer) for t in texts]
    assert not any(boxes[i].overlaps(boxes[j]) for i in range(len(boxes)) for j in range(i+1,len(boxes)))
    assert all(ax.figure.bbox.contains(b.x0,b.y0) and ax.figure.bbox.contains(b.x1,b.y1) for b in boxes)
    assert not any(t.arrow_patch.get_path().intersects_bbox(b,filled=False)
        for i,t in enumerate(texts) if getattr(t,'arrow_patch',None) is not None for j,b in enumerate(boxes) if i!=j)

def draw(kind,layout):
    fig=plt.figure(figsize=FIGURE_SIZES[layout],facecolor='white')
    ax=fig.add_axes(AXES_RECTS[layout])
    column='cell_type_plot' if kind=='cell' else 'stage_standardized'
    palette=CELL_COLORS if kind=='cell' else STAGE_COLORS
    colors=obs[column].map(palette).to_numpy()
    dots=ax.scatter(xy[POINT_ORDER,0],xy[POINT_ORDER,1],c=colors[POINT_ORDER],
        s=POINT_SIZE,alpha=POINT_ALPHA,linewidths=0,edgecolors='none',zorder=1)
    for path in model['plot_paths']:
        ax.plot(path[:,0],path[:,1],color='#191919',lw=CURVE_WIDTH,alpha=.93,zorder=3,solid_capstyle='round')
    ax.set(xlim=X_LIMITS,ylim=Y_LIMITS,xlabel='UMAP1',ylabel='UMAP2')
    ax.set_aspect('auto');ax.set_box_aspect(1)
    ax.set_xticks(X_TICKS);ax.set_yticks(Y_TICKS)
    # A single ASCII string per editable tick, including '-5'.
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value,_:f'{value:g}'))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda value,_:f'{value:g}'))
    ax.spines[['top','right']].set_visible(False)
    for side in ('bottom','left'):ax.spines[side].set_linewidth(1.1)
    ax.tick_params(labelsize=7,width=1.1)
    ax.set_title(TITLE,fontsize=12,pad=10)
    if layout=='legend':
        counts=obs[column].value_counts()
        handles=[Line2D([],[],marker='o',linestyle='',color=palette[key],markersize=5,
            label=f'{key} ({counts[key]:,})') for key in palette if counts.get(key,0)>0]
        ax.legend(handles=handles,loc='center left',bbox_to_anchor=(1.035,.5),fontsize=LEGEND_SIZE,
            frameon=False,handletextpad=.35,labelspacing=.6)
    else:place_labels(ax,CELL_LAYOUT if kind=='cell' else STAGE_LAYOUT)
    fig.canvas.draw();box=ax.get_window_extent();assert abs(box.width-box.height)<.01
    tick_texts=[*ax.get_xticklabels(),*ax.get_yticklabels()]
    assert all(chr(0x2212) not in text.get_text() for text in tick_texts)
    assert [text.get_text() for text in ax.get_xticklabels()]==['-5','0','5','10','15']
    assert len(dots.get_offsets())==7355
    figure_checks.append({'view':kind,'mode':layout,'figsize':list(fig.get_size_inches()),
        'axes_position':list(ax.get_position().bounds),'xlim':list(ax.get_xlim()),
        'ylim':list(ax.get_ylim()),'xticks':list(ax.get_xticks()),'yticks':list(ax.get_yticks()),
        'transform':ax.transData.get_affine().get_matrix().tolist()})
    # Set the generated file path.
    name=str(ROOT)
    # Set the file or directory path.
    fig.savefig(ROOT,dpi=DPI,facecolor='white')
    # Set the file or directory path.
    editable_pdf(fig,ROOT)
    preview=io.BytesIO();fig.savefig(preview,format='png',dpi=INLINE_DPI);plt.close(fig)
    display(Image(data=preview.getvalue(),width=760 if layout=='label' else 950))
    return name

FIGURES.mkdir(parents=True,exist_ok=True)
figure_checks=[]
figure_files=[draw(kind,layout) for kind in ('cell','stage') for layout in ('legend','label')]
assert len(figure_files)==4 and len(obs)==7355
print('Saved four 600-dpi PNGs and four editable vector PDFs; original branches and pseudotime preserved.')
_runtime.cleanup()
if TEMP_PARENT.exists() and not any(TEMP_PARENT.iterdir()):TEMP_PARENT.rmdir()
